In [49]:
import os
import json
import time
import numpy as np
import pandas as pd
import copy
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix

In [52]:
class CFG:
    # данные
    DATA_PATH = "/Users/test/Desktop/GP5/DL_project_5/data/mlp_dataset.csv"
    PRODUCT_PRED = "/Users/test/Desktop/GP5/DL_project_5/data/product_predictions.csv"
    ISSUE_PRED = "/Users/test/Desktop/GP5/DL_project_5/data/issue_predictions.csv"
    TARGET = "Risk" 

    CAT_FEATURES = ["predicted_product", "predicted_issue", "Company", "State", "day_of_week"]
    NUM_FEATURES = ["Complaint_length", "Word_count", "month", "hour", "product_confidence", "issue_confidence"]

    # DagsHub / MLflow
    DAGSHUB_OWNER = "kulikanton05"
    DAGSHUB_REPO = "GP5"

    # разбиение
    TEST_SIZE = 0.2  
    VAL_SIZE = 0.1 
    RANDOM_STATE = 42
    OHE_MIN_FREQ = 0.005  

    # обучение
    BATCH_SIZE = 256
    EPOCHS = 20
    LEARNING_RATE = 5e-4
    DROPOUT = 0.5

def seed_everything(seed=CFG.RANDOM_STATE):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

In [22]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

mps


In [4]:
df = pd.read_csv(CFG.DATA_PATH)
df

,Product,Issue,Company,State,Complaint_length,Word_count,month,day_of_week,hour,Risk
0,Checking or savings account,Managing an account,AMERIS BANCORP,GA,1002,188,1,Thursday,2,0
1,Debt collection,Attempts to collect debt not owed,"Kriya Capital, LLC",GA,2039,332,1,Thursday,3,1
2,Debt collection,Took or threatened to take negative or legal a...,Rosebud Economic Development Corporation,TX,530,85,1,Thursday,0,1
3,Debt collection,Took or threatened to take negative or legal a...,"Portfolio Recovery Associates, LLC",TX,430,74,1,Thursday,0,0
4,Credit card,Other,"EQUIFAX, INC.",IL,182,31,1,Thursday,0,1
...,...,...,...,...,...,...,...,...,...,...
53893,Credit reporting or other personal consumer re...,Incorrect information on your report,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",HI,272,39,5,Wednesday,19,0
53894,Debt collection,Other,"Nationwide Capital Services, LLC",TX,231,39,5,Wednesday,22,0
53895,Credit card,Problem with a purchase shown on your statement,"CITIBANK, N.A.",OH,2281,434,5,Wednesday,23,0
53896,Debt collection,Attempts to collect debt not owed,WELLS FARGO & COMPANY,GA,1704,277,5,Wednesday,22,0


In [6]:
df = df.reset_index(drop=True)
df["row_id"] = df.index
df

,Product,Issue,Company,State,Complaint_length,Word_count,month,day_of_week,hour,Risk,row_id
0,Checking or savings account,Managing an account,AMERIS BANCORP,GA,1002,188,1,Thursday,2,0,0
1,Debt collection,Attempts to collect debt not owed,"Kriya Capital, LLC",GA,2039,332,1,Thursday,3,1,1
2,Debt collection,Took or threatened to take negative or legal a...,Rosebud Economic Development Corporation,TX,530,85,1,Thursday,0,1,2
3,Debt collection,Took or threatened to take negative or legal a...,"Portfolio Recovery Associates, LLC",TX,430,74,1,Thursday,0,0,3
4,Credit card,Other,"EQUIFAX, INC.",IL,182,31,1,Thursday,0,1,4
...,...,...,...,...,...,...,...,...,...,...,...
53893,Credit reporting or other personal consumer re...,Incorrect information on your report,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",HI,272,39,5,Wednesday,19,0,53893
53894,Debt collection,Other,"Nationwide Capital Services, LLC",TX,231,39,5,Wednesday,22,0,53894
53895,Credit card,Problem with a purchase shown on your statement,"CITIBANK, N.A.",OH,2281,434,5,Wednesday,23,0,53895
53896,Debt collection,Attempts to collect debt not owed,WELLS FARGO & COMPANY,GA,1704,277,5,Wednesday,22,0,53896


In [10]:
product_pred = pd.read_csv(CFG.PRODUCT_PRED)
issue_pred = pd.read_csv(CFG.ISSUE_PRED)
product_pred.shape, issue_pred.shape

((53898, 5), (53898, 5))

In [ ]:
df = df.merge(product_pred[["row_id", "predicted_product", "product_confidence"]], on="row_id", how="left")
df = df.merge(issue_pred[["row_id", "predicted_issue", "issue_confidence"]], on="row_id", how="left")
df

,Product,Issue,Company,State,Complaint_length,Word_count,month,day_of_week,hour,Risk,row_id,predicted_product,product_confidence,predicted_issue,issue_confidence
0,Checking or savings account,Managing an account,AMERIS BANCORP,GA,1002,188,1,Thursday,2,0,0,Checking or savings account,0.977000,Managing an account,0.899764
1,Debt collection,Attempts to collect debt not owed,"Kriya Capital, LLC",GA,2039,332,1,Thursday,3,1,1,Debt collection,0.994181,Attempts to collect debt not owed,0.860538
2,Debt collection,Took or threatened to take negative or legal a...,Rosebud Economic Development Corporation,TX,530,85,1,Thursday,0,1,2,Debt collection,0.996244,Took or threatened to take negative or legal a...,0.987370
3,Debt collection,Took or threatened to take negative or legal a...,"Portfolio Recovery Associates, LLC",TX,430,74,1,Thursday,0,0,3,Debt collection,0.997443,Took or threatened to take negative or legal a...,0.988554
4,Credit card,Other,"EQUIFAX, INC.",IL,182,31,1,Thursday,0,1,4,Credit card,0.989423,Other,0.964618
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
53893,Credit reporting or other personal consumer re...,Incorrect information on your report,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",HI,272,39,5,Wednesday,19,0,53893,Credit reporting or other personal consumer re...,0.979107,Incorrect information on your report,0.949851
53894,Debt collection,Other,"Nationwide Capital Services, LLC",TX,231,39,5,Wednesday,22,0,53894,Debt collection,0.993057,Other,0.896801
53895,Credit card,Problem with a purchase shown on your statement,"CITIBANK, N.A.",OH,2281,434,5,Wednesday,23,0,53895,Credit card,0.990808,Problem with a purchase shown on your statement,0.939765
53896,Debt collection,Attempts to collect debt not owed,WELLS FARGO & COMPANY,GA,1704,277,5,Wednesday,22,0,53896,Credit card,0.553421,Incorrect information on your report,0.514651


In [12]:
df['Company'].value_counts()

Company
WELLS FARGO & COMPANY                    2588
CITIBANK, N.A.                           2586
JPMORGAN CHASE & CO.                     2152
BANK OF AMERICA, NATIONAL ASSOCIATION    2136
CAPITAL ONE FINANCIAL CORPORATION        2086
                                         ... 
New Horizons Finance Company, Inc           1
Lentegrity Management, LLC                  1
Metallicus, Inc.                            1
OLD VIRGINIA MORTGAGE, INC                  1
AUDIT SYSTEMS,INC                           1
Name: count, Length: 1417, dtype: int64

In [14]:
X = df[CFG.CAT_FEATURES + CFG.NUM_FEATURES].copy()
y = df[CFG.TARGET].values

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=CFG.TEST_SIZE, random_state=CFG.RANDOM_STATE, stratify=y)
val_relative = CFG.VAL_SIZE / (1.0 - CFG.TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=val_relative, random_state=CFG.RANDOM_STATE, stratify=y_trainval)

In [18]:
len(X_train), len(X_val), len(X_test)

(37728, 5390, 10780)

In [19]:
def save_split_csv(Xpart, ypart, name):
    out = Xpart.copy()
    out[CFG.TARGET] = ypart
    path = f"/Users/test/Desktop/GP5/DL_project_5/artifacts_mlp/risk_{name}.csv"
    out.to_csv(path, index=False)
    return path

SPLIT_PATHS = {
    "train": save_split_csv(X_train, y_train, "train"),
    "val":   save_split_csv(X_val,   y_val,   "val"),
    "test":  save_split_csv(X_test,  y_test,  "test"),
}
print("Сохранены сплиты:", SPLIT_PATHS)

Сохранены сплиты: {'train': '/Users/test/Desktop/GP5/DL_project_5/artifacts_mlp/risk_train.csv', 'val': '/Users/test/Desktop/GP5/DL_project_5/artifacts_mlp/risk_val.csv', 'test': '/Users/test/Desktop/GP5/DL_project_5/artifacts_mlp/risk_test.csv'}


In [20]:
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=CFG.OHE_MIN_FREQ,
                          sparse_output=False), CFG.CAT_FEATURES),
    ("num", StandardScaler(), CFG.NUM_FEATURES),
])

X_train_mat = preprocessor.fit_transform(X_train).astype(np.float32)
X_val_mat   = preprocessor.transform(X_val).astype(np.float32)
X_test_mat  = preprocessor.transform(X_test).astype(np.float32)

input_dim = X_train_mat.shape[1]
input_dim

100

In [21]:
X_train_mat.shape, X_val_mat.shape, X_test_mat.shape

((37728, 100), (5390, 100), (10780, 100))

In [23]:
def make_loader(X_mat, y_arr, shuffle, drop_last=False):
    ds = TensorDataset(
        torch.tensor(X_mat, dtype=torch.float32), 
        torch.tensor(y_arr, dtype=torch.float32).unsqueeze(1)
    )
    return DataLoader(ds, batch_size=CFG.BATCH_SIZE, shuffle=shuffle, drop_last=drop_last)

In [24]:
train_loader = make_loader(X_train_mat, y_train, shuffle=True, drop_last=True)
val_loader   = make_loader(X_val_mat,   y_val,   shuffle=False)
test_loader  = make_loader(X_test_mat,  y_test,  shuffle=False)

In [26]:
pos = float((y_train == 1).sum())
neg = float((y_train == 0).sum())
pos_weight = torch.tensor([neg / pos], dtype=torch.float32, device=device)
pos_weight

tensor([3.5598], device='mps:0')

In [27]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "roc_auc":   roc_auc_score(y_true, y_prob),
    }


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, n, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * yb.size(0)
        n += yb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == yb).sum().item()
    return total_loss / n, correct / n


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss, n, correct = 0.0, 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * yb.size(0)
        n += yb.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == yb).sum().item()
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    probs, trues = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        p = torch.sigmoid(model(xb)).cpu().numpy().ravel()
        probs.extend(p.tolist())
        trues.extend(yb.numpy().ravel().tolist())
    probs = np.array(probs); trues = np.array(trues)
    preds = (probs >= 0.5).astype(int)
    return compute_metrics(trues, preds, probs), trues, preds, probs


In [28]:
import dagshub
import mlflow

dagshub.init(repo_owner=CFG.DAGSHUB_OWNER, repo_name=CFG.DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("Risk_Classification")
print("Tracking URI:", mlflow.get_tracking_uri())

/Users/test/Desktop/GP5/DL_project_5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as kulikanton05

Initialized MLflow to track repo "kulikanton05/GP5"

Repository kulikanton05/GP5 initialized!

2026/06/16 07:24:40 INFO mlflow.tracking.fluent: Experiment with name 'Risk_Classification' does not exist. Creating a new experiment.


Tracking URI: https://dagshub.com/kulikanton05/GP5.mlflow


In [53]:
def run_experiment(model, model_name, hidden_layers, dropout=CFG.DROPOUT, epochs=CFG.EPOCHS, lr=CFG.LEARNING_RATE):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params({
            "model_name":    model_name,
            "hidden_layers": str(hidden_layers),
            "activation":    "ReLU",
            "dropout":       dropout,
            "batch_size":    CFG.BATCH_SIZE,
            "learning_rate": lr,
            "epochs":        epochs,
            "input_dim":     input_dim,
        })

        for epoch in range(1, epochs + 1):
            tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
            va_loss, va_acc = validate_epoch(model, val_loader, criterion)
            mlflow.log_metrics({"train_loss": tr_loss, "val_loss": va_loss, "train_acc": tr_acc, "val_acc": va_acc}, step=epoch)
            if va_loss < best_val_loss:
                best_val_loss = va_loss
                best_epoch = epoch
                best_state = copy.deepcopy(model.state_dict())
            if epoch % 5 == 0 or epoch == 1:
                print(f"[{model_name}] epoch {epoch:2d}/{epochs} "
                      f"| train_loss {tr_loss:.4f} acc {tr_acc:.3f} "
                      f"| val_loss {va_loss:.4f} acc {va_acc:.3f}")

        model.load_state_dict(best_state)
        print(
            f"\n[{model_name}] "
            f"Best epoch = {best_epoch}, "
            f"best val_loss = {best_val_loss:.4f}"
        )

        test_metrics, y_true, y_pred, y_prob = evaluate_model(model, test_loader)
        mlflow.log_metrics({
            "accuracy":  test_metrics["accuracy"],
            "precision": test_metrics["precision"],
            "recall":    test_metrics["recall"],
            "f1":        test_metrics["f1"],
            "roc_auc":   test_metrics["roc_auc"],
        })
        print(f"\n[{model_name}] TEST:", {k: round(v, 4) for k, v in test_metrics.items()})

        rep = classification_report(y_true, y_pred, target_names=["Risk=0", "Risk=1"], zero_division=0)
        rep_path = f"/Users/test/Desktop/GP5/DL_project_5/artifacts_mlp/{model_name}_report.txt"
        with open(rep_path, "w") as f:
            f.write(rep)
        mlflow.log_artifact(rep_path, artifact_path="reports")

        mlflow.pytorch.log_model(model, artifact_path="model")

        for p in SPLIT_PATHS.values():
            mlflow.log_artifact(p, artifact_path="data")

    return {"test_metrics": test_metrics, "y_true": y_true, "y_pred": y_pred, "y_prob": y_prob}


In [30]:
class MLP_first(nn.Module):
    def __init__(self, input_dim, dropout=CFG.DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.out(x)  


In [ ]:
seed_everything()
res_first = run_experiment(MLP_first(input_dim), "MLP_first", hidden_layers=[64])


[MLP_first] epoch  1/30 | train_loss 1.0302 acc 0.604 | val_loss 0.9658 acc 0.702
[MLP_first] epoch  5/30 | train_loss 0.8798 acc 0.700 | val_loss 0.8814 acc 0.705
[MLP_first] epoch 10/30 | train_loss 0.8639 acc 0.706 | val_loss 0.8743 acc 0.711
[MLP_first] epoch 15/30 | train_loss 0.8562 acc 0.708 | val_loss 0.8716 acc 0.706
[MLP_first] epoch 20/30 | train_loss 0.8509 acc 0.711 | val_loss 0.8717 acc 0.707
[MLP_first] epoch 25/30 | train_loss 0.8442 acc 0.715 | val_loss 0.8723 acc 0.698
[MLP_first] epoch 30/30 | train_loss 0.8385 acc 0.713 | val_loss 0.8786 acc 0.703

[MLP_first] TEST: {'accuracy': 0.7001, 'precision': 0.3891, 'recall': 0.6451, 'f1': 0.4854, 'roc_auc': 0.7625}


2026/06/16 07:59:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:00:01 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run MLP_first at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2/runs/9f387df3f2e843dbb5d708a8194537ff
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2


In [36]:
class MLP_second(nn.Module):
    def __init__(self, input_dim, dropout=CFG.DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.out(x)

In [38]:
seed_everything()
res_sec = run_experiment(MLP_second(input_dim), "MLP_second", hidden_layers=[128, 64])

[MLP_second] epoch  1/30 | train_loss 1.0052 acc 0.683 | val_loss 0.9088 acc 0.703
[MLP_second] epoch  5/30 | train_loss 0.8681 acc 0.700 | val_loss 0.8755 acc 0.689
[MLP_second] epoch 10/30 | train_loss 0.8426 acc 0.708 | val_loss 0.8751 acc 0.700
[MLP_second] epoch 15/30 | train_loss 0.8232 acc 0.710 | val_loss 0.8884 acc 0.712
[MLP_second] epoch 20/30 | train_loss 0.8066 acc 0.711 | val_loss 0.8920 acc 0.693
[MLP_second] epoch 25/30 | train_loss 0.7923 acc 0.714 | val_loss 0.9079 acc 0.678
[MLP_second] epoch 30/30 | train_loss 0.7778 acc 0.710 | val_loss 0.9098 acc 0.655

[MLP_second] TEST: {'accuracy': 0.651, 'precision': 0.3549, 'recall': 0.7234, 'f1': 0.4762, 'roc_auc': 0.7567}


2026/06/16 08:02:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:03:00 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run MLP_second at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2/runs/0ca2513465d240da9380486a3393d749
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2


In [40]:
class MLP_third(nn.Module):
    def __init__(self, input_dim, dropout=CFG.DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.fc3 = nn.Linear(128, 64)
        self.out = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.dropout(self.relu(self.bn1(self.fc1(x))))
        x = self.dropout(self.relu(self.bn2(self.fc2(x))))
        x = self.dropout(self.relu(self.fc3(x)))
        return self.out(x)


In [54]:
seed_everything()
res_3 = run_experiment(MLP_third(input_dim), "MLP_3_v2", hidden_layers=[256, 128, 64])


[MLP_3_v2] epoch  1/20 | train_loss 1.0154 acc 0.610 | val_loss 0.9123 acc 0.672
[MLP_3_v2] epoch  5/20 | train_loss 0.8740 acc 0.698 | val_loss 0.8749 acc 0.696
[MLP_3_v2] epoch 10/20 | train_loss 0.8469 acc 0.706 | val_loss 0.8734 acc 0.698
[MLP_3_v2] epoch 15/20 | train_loss 0.8303 acc 0.712 | val_loss 0.8765 acc 0.695
[MLP_3_v2] epoch 20/20 | train_loss 0.8143 acc 0.715 | val_loss 0.8804 acc 0.701

[MLP_3_v2] Best epoch = 11, best val_loss = 0.8696

[MLP_3_v2] TEST: {'accuracy': 0.703, 'precision': 0.3937, 'recall': 0.6565, 'f1': 0.4922, 'roc_auc': 0.7637}


2026/06/16 08:36:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:36:16 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run MLP_3_v2 at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2/runs/42d78acae5c24895afa3cc217fb789f5
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2


In [43]:
class MLP_first_v2(nn.Module):
    def __init__(self, input_dim, dropout=CFG.DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(64, 1)
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.dropout(x)
        return self.out(x)

In [51]:
seed_everything()

res_first_v2 = run_experiment(MLP_first_v2(input_dim), "MLP_first_v3", hidden_layers=[64])

[MLP_first_v3] epoch  1/15 | train_loss 1.0395 acc 0.550 | val_loss 0.9690 acc 0.692
[MLP_first_v3] epoch  5/15 | train_loss 0.8816 acc 0.691 | val_loss 0.8813 acc 0.698
[MLP_first_v3] epoch 10/15 | train_loss 0.8631 acc 0.700 | val_loss 0.8757 acc 0.688
[MLP_first_v3] epoch 15/15 | train_loss 0.8551 acc 0.703 | val_loss 0.8736 acc 0.705

[MLP_first_v3] Best epoch = 15, best val_loss = 0.8736

[MLP_first_v3] TEST: {'accuracy': 0.7106, 'precision': 0.3999, 'recall': 0.6387, 'f1': 0.4919, 'roc_auc': 0.7646}


2026/06/16 08:31:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 08:31:27 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run MLP_first_v3 at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2/runs/872a279995ea4542aae7b316b9572eff
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2


In [57]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=CFG.DROPOUT):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        identity = x         
        out = self.dropout(self.relu(self.bn1(self.fc1(x))))
        out = self.bn2(self.fc2(out))
        out = out + identity   
        return self.relu(out)


class ResidualMLP(nn.Module):
    def __init__(self, input_dim, hidden=64, dropout=CFG.DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden) 
        self.block1 = ResidualBlock(hidden, dropout)
        self.block2 = ResidualBlock(hidden, dropout)
        self.out = nn.Linear(hidden, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.input_proj(x))
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x)


In [58]:
seed_everything()
res_residual = run_experiment(ResidualMLP(input_dim), "ResidualMLP", hidden_layers=[128, 128, 128])


[ResidualMLP] epoch  1/20 | train_loss 1.0681 acc 0.541 | val_loss 0.9982 acc 0.633
[ResidualMLP] epoch  5/20 | train_loss 0.8855 acc 0.683 | val_loss 0.8836 acc 0.683
[ResidualMLP] epoch 10/20 | train_loss 0.8594 acc 0.697 | val_loss 0.8767 acc 0.680
[ResidualMLP] epoch 15/20 | train_loss 0.8370 acc 0.700 | val_loss 0.8805 acc 0.681
[ResidualMLP] epoch 20/20 | train_loss 0.8204 acc 0.710 | val_loss 0.8871 acc 0.677

[ResidualMLP] Best epoch = 10, best val_loss = 0.8767

[ResidualMLP] TEST: {'accuracy': 0.6866, 'precision': 0.3814, 'recall': 0.6899, 'f1': 0.4913, 'roc_auc': 0.7643}


2026/06/16 09:24:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/16 09:24:12 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run ResidualMLP at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2/runs/d69daa6c7d5a49eba6fce9d7ba68ccc5
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/2
